# BYV-AI — BYV-FP v0.2.1 Held-out Test Evaluation

**Base model:** `Qwen/Qwen3-4B-Thinking-2507`  
**Adapter:** output dari notebook training BYV-FP v0.2.1  
**Test split:** 35 held-out records, never used for training.  

Notebook ini memakai dataset ZIP yang sama dari Google Drive. Pada training pertama, ZIP disimpan ke Drive; evaluasi akan memakai file tersebut dan hanya membuka `test.jsonl`. Notebook ini tidak melatih model dan tidak mengeksekusi tool. Output akan disimpan ke Google Drive untuk pemeriksaan manual.

In [ ]:
%pip install -q -U "transformers>=4.51.0" "accelerate>=1.0.0" "peft>=0.14.0" "bitsandbytes>=0.45.0" "safetensors>=0.4.5"

In [ ]:
import os, sys, json, re, time, csv, hashlib, urllib.request, inspect, io, zipfile
from pathlib import Path
from collections import Counter
import torch, transformers
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel
print("Python:", sys.version.split()[0], "| Torch:", torch.__version__, "| Transformers:", transformers.__version__)

In [ ]:
MODEL_ID = "Qwen/Qwen3-4B-Thinking-2507"
MODEL_REVISION = "768f209d9ea81521153ed38c47d515654e938aea"
DATASET_VERSION = "byv-fp-v0.2.1"
RUN_NAME = "byv-ai-qwen3-4b-thinking-2507-byvfp-qlora-t4-fp16-v0.2.1"
EXPECTED_TEST_SHA256 = "381c359575f470cf7d05de92c369169fdf5f17624a2062508723f29dc0aba488"
ZIP_NAME = "BYV-AI_dataset_BYV-FP_v0.2.1.zip"
ZIP_ROOT = "BYV-AI_dataset_byvfp_v0.2.1/"
MAX_NEW_TOKENS = 2048
RUN_BASELINE = False  # set True to also compare the unadapted base model

from google.colab import drive, files
drive.mount('/content/drive')
DRIVE_ROOT = Path('/content/drive/MyDrive/BYV-AI')
DATA_DIR = DRIVE_ROOT / 'data' / DATASET_VERSION
ADAPTER_DIR = DRIVE_ROOT / 'outputs' / RUN_NAME
RESULT_DIR = DRIVE_ROOT / 'experiments' / RUN_NAME / 'test_evaluation'
DATA_DIR.mkdir(parents=True, exist_ok=True)
RESULT_DIR.mkdir(parents=True, exist_ok=True)

if not torch.cuda.is_available():
    raise RuntimeError("Pilih Runtime > Change runtime type > GPU.")
if not (ADAPTER_DIR / 'adapter_config.json').exists():
    raise FileNotFoundError(f"Adapter tidak ditemukan: {ADAPTER_DIR}. Jalankan notebook training lebih dulu.")

ZIP_PATH = DATA_DIR / ZIP_NAME
if not ZIP_PATH.exists():
    print(f"Upload {ZIP_NAME} sekali saja. Salinannya akan disimpan di Google Drive.")
    uploaded = files.upload()
    if ZIP_NAME not in uploaded:
        raise FileNotFoundError(f"Please upload the exact file: {ZIP_NAME}")
    ZIP_PATH.write_bytes(uploaded[ZIP_NAME])
    del uploaded
else:
    print("Dataset ZIP ditemukan di Google Drive; memakai salinan yang sudah ada.")

with zipfile.ZipFile(ZIP_PATH) as archive:
    member = ZIP_ROOT + 'test.jsonl'
    if member not in archive.namelist():
        raise FileNotFoundError(f"Dataset ZIP is missing {member}")
    test_bytes = archive.read(member)  # held-out split only
actual_hash = hashlib.sha256(test_bytes).hexdigest()
if actual_hash != EXPECTED_TEST_SHA256:
    raise RuntimeError(f"Test SHA-256 mismatch: {actual_hash} != {EXPECTED_TEST_SHA256}")
TEST_PATH = DATA_DIR / 'test.jsonl'
TEST_PATH.write_bytes(test_bytes)
del test_bytes
print("Verified held-out test SHA-256:", actual_hash)
print("Adapter:", ADAPTER_DIR)

In [ ]:
HEADER = "BYV-FP/1.0"
ID_RE = re.compile(r"^[A-Za-z0-9][A-Za-z0-9._:-]{0,63}$")
TOOL_RE = re.compile(r"^[A-Za-z_][A-Za-z0-9_.:-]{0,127}$")

def load_jsonl(path):
    with open(path, encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]

def active_tools(messages):
    system = next(m["content"] for m in messages if m["role"] == "system")
    marker = "ACTIVE_TOOLS_JSON:\n"
    if marker not in system:
        return {}
    raw = system.split(marker, 1)[1].split("\nUse the schemas above", 1)[0]
    return {x["name"]: x["input_schema"] for x in json.loads(raw)}

def schema_errors(value, schema, path="$" ):
    errors=[]; typ=schema.get("type")
    checks={"object":lambda x:isinstance(x,dict),"array":lambda x:isinstance(x,list),"string":lambda x:isinstance(x,str),"integer":lambda x:isinstance(x,int) and not isinstance(x,bool),"number":lambda x:isinstance(x,(int,float)) and not isinstance(x,bool),"boolean":lambda x:isinstance(x,bool),"null":lambda x:x is None}
    if typ and (typ not in checks or not checks[typ](value)):
        return [f"{path}: expected {typ}, got {type(value).__name__}"]
    if isinstance(value,dict):
        props=schema.get("properties",{})
        for key in schema.get("required",[]):
            if key not in value: errors.append(f"{path}.{key}: missing required property")
        for key,item in value.items():
            if key not in props:
                if schema.get("additionalProperties",True) is False: errors.append(f"{path}.{key}: undeclared property")
            else: errors.extend(schema_errors(item,props[key],f"{path}.{key}"))
    if isinstance(value,list) and "items" in schema:
        for i,item in enumerate(value): errors.extend(schema_errors(item,schema["items"],f"{path}[{i}]"))
    if isinstance(value,(int,float)) and not isinstance(value,bool):
        if "minimum" in schema and value<schema["minimum"]: errors.append(f"{path}: below minimum")
        if "maximum" in schema and value>schema["maximum"]: errors.append(f"{path}: above maximum")
    if isinstance(value,str):
        if "minLength" in schema and len(value)<schema["minLength"]: errors.append(f"{path}: shorter than minLength")
        if "maxLength" in schema and len(value)>schema["maxLength"]: errors.append(f"{path}: longer than maxLength")
    return errors

def validate_prediction(text,messages):
    if not text.startswith(HEADER+"\n"):
        return {"is_byv_fp":False,"valid":True,"errors":[]}
    try:
        env=json.loads(text.split("\n",1)[1])
    except Exception as exc:
        return {"is_byv_fp":True,"valid":False,"errors":[f"invalid JSON: {exc}"]}
    errors=[]
    if set(env)!={"type","calls"} or env.get("type")!="tool_calls":
        errors.append("invalid top-level envelope")
        return {"is_byv_fp":True,"valid":False,"errors":errors}
    calls=env.get("calls")
    if not isinstance(calls,list) or not (1<=len(calls)<=32):
        errors.append("calls must be a non-empty array <= 32")
        return {"is_byv_fp":True,"valid":False,"errors":errors}
    tools=active_tools(messages); seen=set()
    for i,call in enumerate(calls):
        if not isinstance(call,dict) or set(call)!={"id","tool","arguments"}:
            errors.append(f"calls[{i}]: invalid keys")
            continue
        call_id,tool,args=call["id"],call["tool"],call["arguments"]
        if not isinstance(call_id,str) or not ID_RE.fullmatch(call_id): errors.append(f"calls[{i}]: invalid id")
        if call_id in seen: errors.append(f"calls[{i}]: duplicate id")
        seen.add(call_id)
        if not isinstance(tool,str) or not TOOL_RE.fullmatch(tool): errors.append(f"calls[{i}]: invalid tool name"); continue
        if tool not in tools: errors.append(f"calls[{i}]: unknown tool {tool}"); continue
        if not isinstance(args,dict): errors.append(f"calls[{i}]: arguments must be object"); continue
        errors.extend(schema_errors(args,tools[tool],f"calls[{i}].arguments"))
    return {"is_byv_fp":True,"valid":len(errors)==0,"errors":errors}

def final_only_content(content):
    if "</think>" in content or "<think>" in content: return content
    return "</think>\n"+content

def normalize_context_messages(messages):
    out=[]
    for msg in messages:
        item={"role":msg["role"],"content":msg["content"]}
        if item["role"]=="assistant": item["content"]=final_only_content(item["content"])
        out.append(item)
    return out

def extract_final_channel(raw):
    if "</think>" in raw: return raw.rsplit("</think>",1)[1].strip()
    if "<think>" in raw: return "[NO_FINAL_CHANNEL_DELIMITER]"
    return raw.strip()

test_records=load_jsonl(TEST_PATH)
if len(test_records)!=35: print("Warning: expected 35 held-out records; found",len(test_records))
print("Held-out test records:",len(test_records))

In [ ]:
cap = torch.cuda.get_device_capability(0)
COMPUTE_DTYPE = torch.bfloat16 if cap[0] >= 8 and torch.cuda.is_bf16_supported() else torch.float16
bnb_config = BitsAndBytesConfig(load_in_4bit=True,bnb_4bit_quant_type="nf4",bnb_4bit_compute_dtype=COMPUTE_DTYPE,bnb_4bit_use_double_quant=True)
TOKENIZER = AutoTokenizer.from_pretrained(ADAPTER_DIR if (ADAPTER_DIR/"tokenizer_config.json").exists() else MODEL_ID,use_fast=True)
if TOKENIZER.pad_token_id is None: TOKENIZER.pad_token=TOKENIZER.eos_token
major=int(transformers.__version__.split('.')[0])
dtype_key="dtype" if major>=5 else "torch_dtype"
kwargs={"revision":MODEL_REVISION,"quantization_config":bnb_config,"device_map":"auto","low_cpu_mem_usage":True}
kwargs[dtype_key]=COMPUTE_DTYPE
base_model=AutoModelForCausalLM.from_pretrained(MODEL_ID,**kwargs)
BASE_REVISION=getattr(base_model.config,"_commit_hash",None)
base_model.eval()
print("Loaded base:",MODEL_ID,"revision:",BASE_REVISION,"dtype:",COMPUTE_DTYPE)

In [ ]:
def generate_final(model_obj,messages,max_new_tokens=MAX_NEW_TOKENS):
    prompt_messages=normalize_context_messages(messages)
    prompt=TOKENIZER.apply_chat_template(prompt_messages,tokenize=False,add_generation_prompt=True)
    device=model_obj.get_input_embeddings().weight.device
    inputs=TOKENIZER(prompt,return_tensors="pt").to(device)
    model_obj.config.use_cache=True
    start=time.time()
    with torch.inference_mode():
        output=model_obj.generate(**inputs,max_new_tokens=max_new_tokens,do_sample=True,temperature=0.6,top_p=0.95,top_k=20,pad_token_id=TOKENIZER.pad_token_id,eos_token_id=TOKENIZER.eos_token_id,use_cache=True)
    latency=round(time.time()-start,3)
    continuation=TOKENIZER.decode(output[0][inputs["input_ids"].shape[1]:],skip_special_tokens=True)
    model_obj.config.use_cache=False
    return extract_final_channel(continuation),latency

def run_eval(model_obj,tag):
    results=[]
    for i,row in enumerate(test_records,1):
        if row["messages"][-1]["role"]!="assistant":
            print("Skipping non-assistant final record:",row.get("id")); continue
        prompt_messages=row["messages"][:-1]
        reference=row["messages"][-1]["content"]
        pred,latency=generate_final(model_obj,prompt_messages)
        check=validate_prediction(pred,prompt_messages)
        ref_byv=reference.startswith(HEADER+"\n")
        result={
            "id":row["id"],"category":row.get("category"),
            "reference":reference,"prediction":pred,
            "reference_byv_fp":ref_byv,"prediction_byv_fp":check["is_byv_fp"],
            "byv_fp_valid":check["valid"],"byv_fp_errors":check["errors"],
            "byv_fp_behavior_match":ref_byv==check["is_byv_fp"],
            "latency_seconds":latency,"model_variant":tag,
        }
        results.append(result)
        print(f"[{i:02d}/{len(test_records)}] {row['id']} | {row.get('category')} | {latency}s | BYV-FP={check['is_byv_fp']} | schema_valid={check['valid']}")
    return results

if RUN_BASELINE:
    baseline_results=run_eval(base_model,"base")
    (RESULT_DIR/"test_predictions_base.jsonl").write_text(''.join(json.dumps(x,ensure_ascii=False)+"\n" for x in baseline_results),encoding="utf-8")
else:
    baseline_results=[]

torch.cuda.empty_cache()

In [ ]:
# Load the LoRA adapter on the same 4-bit base model.
model=PeftModel.from_pretrained(base_model,str(ADAPTER_DIR),is_trainable=False)
model.eval()
adapter_results=run_eval(model,"adapter")

(RESULT_DIR/"test_predictions_adapter.jsonl").write_text(
    ''.join(json.dumps(x,ensure_ascii=False)+"\n" for x in adapter_results),encoding="utf-8"
)

fields=["model_variant","id","category","reference_byv_fp","prediction_byv_fp","byv_fp_valid","byv_fp_behavior_match","latency_seconds","byv_fp_errors","reference","prediction"]
all_results=baseline_results+adapter_results
with open(RESULT_DIR/"manual_review.csv","w",newline="",encoding="utf-8-sig") as f:
    writer=csv.DictWriter(f,fieldnames=fields,extrasaction="ignore")
    writer.writeheader();writer.writerows(all_results)

def summarize(rows):
    predicted=[r for r in rows if r["prediction_byv_fp"]]
    return {
        "examples":len(rows),
        "nonempty_predictions":sum(bool(r["prediction"].strip()) for r in rows),
        "reference_byv_fp":sum(r["reference_byv_fp"] for r in rows),
        "prediction_byv_fp":len(predicted),
        "byv_fp_presence_matches":sum(r["byv_fp_behavior_match"] for r in rows),
        "byv_fp_schema_valid":sum(bool(r["byv_fp_valid"]) for r in predicted),
        "byv_fp_schema_invalid":sum(not bool(r["byv_fp_valid"]) for r in predicted),
        "mean_latency_seconds":round(sum(r["latency_seconds"] for r in rows)/max(1,len(rows)),3),
    }

summary={
    "base_model":MODEL_ID,"requested_model_revision":MODEL_REVISION,"base_revision":BASE_REVISION,"adapter_dir":str(ADAPTER_DIR),
    "test_sha256":EXPECTED_TEST_SHA256,"test_examples":len(test_records),
    "adapter":summarize(adapter_results),
    "baseline":summarize(baseline_results) if RUN_BASELINE else None,
    "warning":"Structural/schema validity is not semantic accuracy, safety, grounding, or tool execution. No tool is executed by this notebook.",
}
(RESULT_DIR/"test_summary.json").write_text(json.dumps(summary,ensure_ascii=False,indent=2)+"\n",encoding="utf-8")
print(json.dumps(summary,ensure_ascii=False,indent=2))
print("Saved outputs to:",RESULT_DIR)

## Hasil dan batas interpretasi

Periksa `manual_review.csv`, `test_predictions_adapter.jsonl`, dan `test_summary.json` di Google Drive. Metrik BYV-FP presence dan schema pass hanya mengecek bentuk, tool name, required fields, tipe, dan properti tambahan. Metrik tersebut **tidak** membuktikan semantik argumen, kebenaran jawaban, safety, atau eksekusi tool. Semua tool-call tetap harus diuji lewat SDK dan host gateway.